# SoftGear Phase 4 — MVP Training on Colab

Gear config: `[1, 2, 3, 4]` (B1 baseline)

Go/No-Go: puzzle_accuracy >= 70%

In [ ]:
# Environment setup
!pip install -q torch hydra-core omegaconf wandb numpy pandas

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone repo + prepare data
import os

REPO_DIR = "/content/softgear"
DATA_CACHE = "/content/drive/MyDrive/softgear-data/sudoku-extreme"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/byExist/softgear.git {REPO_DIR}

os.chdir(REPO_DIR)

# Link cached data from Drive (or download)
DATA_DIR = os.path.join(REPO_DIR, "data", "sudoku-extreme")
if os.path.exists(DATA_CACHE):
    os.makedirs("data", exist_ok=True)
    if not os.path.exists(DATA_DIR):
        os.symlink(DATA_CACHE, DATA_DIR)
    print("Data linked from Drive cache")
else:
    print("Download data manually or use HuggingFace Hub:")
    print("  huggingface-cli download <dataset> --local-dir", DATA_DIR)

In [ ]:
# GPU check
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")

In [ ]:
# wandb login
import wandb

wandb.login()

In [ ]:
# B1: Train with gear_sizes=[1,2,3,4]
!python pretrain.py \
    model.gear_sizes="[1,2,3,4]" \
    training.checkpoint_dir="/content/drive/MyDrive/softgear-checkpoints/B1" \
    wandb.project=softgear \
    wandb.entity=null

In [ ]:
# Resume training (after Colab reconnect)
CKPT_DIR = "/content/drive/MyDrive/softgear-checkpoints/B1"

!python pretrain.py \
    model.gear_sizes="[1,2,3,4]" \
    training.checkpoint_dir={CKPT_DIR} \
    resume={CKPT_DIR}/latest.pt \
    wandb.project=softgear \
    wandb.entity=null

In [ ]:
# Evaluate best checkpoint
CKPT_DIR = "/content/drive/MyDrive/softgear-checkpoints/B1"

!python evaluate.py \
    model.gear_sizes="[1,2,3,4]" \
    checkpoint={CKPT_DIR}/best.pt

In [ ]:
# --- Additional experiments (uncomment as needed) ---

# B2: gear_sizes=[1,2,4,8]
# !python pretrain.py \
#     model.gear_sizes="[1,2,4,8]" \
#     training.checkpoint_dir="/content/drive/MyDrive/softgear-checkpoints/B2" \
#     wandb.project=softgear

# B3: gear_sizes=[2,2,2,2] (uniform baseline)
# !python pretrain.py \
#     model.gear_sizes="[2,2,2,2]" \
#     training.checkpoint_dir="/content/drive/MyDrive/softgear-checkpoints/B3" \
#     wandb.project=softgear

# C: Ablation — no adaptive halt
# !python pretrain.py \
#     model.gear_sizes="[1,2,3,4]" \
#     model.adaptive_halt.enabled=false \
#     training.checkpoint_dir="/content/drive/MyDrive/softgear-checkpoints/C" \
#     wandb.project=softgear